# 02: Neural Collaborative Filtering & Two-Tower Architecture in PyTorch

**Track 11: Recommendation Systems** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Deep learning recommendation architectures: User Tower and Item Tower neural embeddings, dot product similarity, and Binary Cross-Entropy click-through optimization.


## 1. Two-Tower Architecture Mechanics
$$\hat{y}_{u, i} = \sigma\left(f_{\text{user}}(u)^T f_{\text{item}}(i)\right)$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerRecommender(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=32):
        super().__init__()
        # User Tower
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.user_fc = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
        
        # Item Tower
        self.item_embed = nn.Embedding(num_items, embed_dim)
        self.item_fc = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
        
    def forward(self, user_ids, item_ids):
        u_emb = self.user_fc(self.user_embed(user_ids))
        i_emb = self.item_fc(self.item_embed(item_ids))
        
        u_norm = F.normalize(u_emb, p=2, dim=-1)
        i_norm = F.normalize(i_emb, p=2, dim=-1)
        
        score = (u_norm * i_norm).sum(dim=-1)
        return torch.sigmoid(score)

model = TwoTowerRecommender(num_users=1000, num_items=500, embed_dim=32)
user_batch = torch.tensor([1, 4, 12, 50])
item_batch = torch.tensor([10, 25, 120, 300])

scores = model(user_batch, item_batch)
print("=== Two-Tower Model Forward Pass ===")
print(f"Predicted Interaction Scores: {scores.detach().numpy().round(4)}")